# Geração de Dados Simulados - Versão 2

## Por que uma versão 2?

Na versão anterior (v1), identifiquei dois problemas que comprometiam a qualidade do modelo de ML:

1. **Dados sem correlação** — as variáveis eram geradas de forma independente, sem conversar entre si. Por exemplo, um cliente com restrição ativa podia ter score 900, o que não reflete a realidade.

2. **Desbalanceamento de classes** — apenas 10% dos clientes eram inadimplentes, fazendo o modelo aprender mal a identificar riscos. O recall da classe 1 ficou em 20%.

## O que mudamos

- Dados gerados em 3 perfis correlacionados: bom pagador, intermediário e risco
- Volume aumentado de 500 para 5.000 clientes
- Variáveis que conversam entre si, refletindo comportamentos reais
- Distribuição: 55% bom pagador, 25% intermediário, 20% risco

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()

server = os.getenv('SQL_SERVER')
database = os.getenv('SQL_DATABASE')

connection_string = f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = create_engine(connection_string)

try:
    with engine.connect() as conn:
        print(f"Conectado com sucesso ao banco: {database}")
        print(f"Servidor: {server}")
except Exception as e:
    print(f"Erro na conexão: {e}")

In [ ]:
from sqlalchemy import text

with engine.connect() as conn:
    conn.execute(text("DELETE FROM decisoes"))
    conn.execute(text("DELETE FROM solicitacoes_credito"))
    conn.execute(text("DELETE FROM analistas"))
    conn.execute(text("DELETE FROM clientes"))
    conn.commit()
    print("Tabelas limpas com sucesso!")

In [ ]:
# Definição dos perfis de clientes
perfis = {
    'bom_pagador': {
        'peso': 0.55,
        'score_credito': (600, 1000),
        'renda_mensal': (3000, 15000),
        'tempo_emprego_anos': (3, 20),
        'qtd_emprestimos_ativos': (0, 2),
        'historico_inadimplencia': 0.05,
        'possui_restricao': 0.01,
        'tem_imovel': 0.60,
        'tem_veiculo': 0.65,
        'chance_inadimplir': 0.05
    },
    'intermediario': {
        'peso': 0.25,
        'score_credito': (500, 700),
        'renda_mensal': (2500, 5000),
        'tempo_emprego_anos': (1, 8),
        'qtd_emprestimos_ativos': (0, 3),
        'historico_inadimplencia': 0.35,
        'possui_restricao': 0.10,
        'tem_imovel': 0.30,
        'tem_veiculo': 0.40,
        'chance_inadimplir': 0.30
    },
    'risco': {
        'peso': 0.20,
        'score_credito': (0, 450),
        'renda_mensal': (800, 2000),
        'tempo_emprego_anos': (0, 3),
        'qtd_emprestimos_ativos': (1, 5),
        'historico_inadimplencia': 0.80,
        'possui_restricao': 0.70,
        'tem_imovel': 0.10,
        'tem_veiculo': 0.15,
        'chance_inadimplir': 0.75
    }
}

print("Perfis definidos com sucesso!")
for nome, config in perfis.items():
    total = int(5000 * config['peso'])
    print(f"  {nome}: {total} clientes")

In [ ]:
fake = Faker('pt_BR')
random.seed(42)
np.random.seed(42)

def gerar_clientes(total=5000):
    clientes = []
    nomes_perfis = list(perfis.keys())
    pesos = [perfis[p]['peso'] for p in nomes_perfis]

    for _ in range(total):
        # Sorteia o perfil baseado nos pesos
        perfil_nome = random.choices(nomes_perfis, weights=pesos, k=1)[0]
        p = perfis[perfil_nome]

        # Gera as variáveis baseadas no perfil
        score = random.randint(*p['score_credito'])
        renda = round(random.uniform(*p['renda_mensal']), 2)
        tempo_emprego = round(random.uniform(*p['tempo_emprego_anos']), 1)
        qtd_emprestimos = random.randint(*p['qtd_emprestimos_ativos'])
        historico_inadimplencia = 1 if random.random() < p['historico_inadimplencia'] else 0
        possui_restricao = 1 if random.random() < p['possui_restricao'] else 0
        tem_imovel = 1 if random.random() < p['tem_imovel'] else 0
        tem_veiculo = 1 if random.random() < p['tem_veiculo'] else 0
        inadimplente = 1 if random.random() < p['chance_inadimplir'] else 0

        # Valor solicitado baseado na renda
        valor_solicitado = round(random.uniform(renda * 0.5, renda * 5), 2)
        prazo_meses = random.choice([12, 24, 36, 48, 60])

        clientes.append({
            'perfil': perfil_nome,
            'nome': fake.name(),
            'cpf': fake.cpf(),
            'idade': random.randint(18, 75),
            'renda_mensal': renda,
            'score_credito': score,
            'tem_imovel': tem_imovel,
            'tem_veiculo': tem_veiculo,
            'tempo_emprego_anos': tempo_emprego,
            'qtd_emprestimos_ativos': qtd_emprestimos,
            'historico_inadimplencia': historico_inadimplencia,
            'possui_restricao': possui_restricao,
            'valor_solicitado': valor_solicitado,
            'prazo_meses': prazo_meses,
            'inadimplente': inadimplente
        })

    return pd.DataFrame(clientes)

df = gerar_clientes(5000)
print(f"Clientes gerados: {len(df)}")
print(df.head())

In [ ]:
# Validação dos dados gerados
print("=== DISTRIBUIÇÃO DOS PERFIS ===")
print(df['perfil'].value_counts())
print(f"\nTotal: {len(df)} clientes")

print("\n=== MÉDIAS POR PERFIL ===")
print(df.groupby('perfil')[['score_credito', 'renda_mensal', 'tempo_emprego_anos']].mean().round(2))

print("\n=== TAXA DE INADIMPLÊNCIA POR PERFIL ===")
print(df.groupby('perfil')['inadimplente'].mean().round(2))

print("\n=== TOTAL DE INADIMPLENTES ===")
print(df['inadimplente'].value_counts())
print(f"Taxa geral de inadimplência: {df['inadimplente'].mean():.2%}")

In [ ]:
# Salvar no SQL Server na tabela clientes_v2
try:
    df.to_sql('clientes_v2', engine, if_exists='replace', index=False)
    print(f"Dados salvos com sucesso na tabela clientes_v2!")
    print(f"Total de registros: {len(df)}")
except Exception as e:
    print(f"Erro ao salvar: {e}")